In [1]:
# imports

import os
import sys
from pathlib import Path
import json
import traceback
import re
from dotenv import load_dotenv
import csv
import random
import unicodedata
from collections import defaultdict, OrderedDict
import pandas as pd

from entities.document import Document
from entities.frames import *  # full coverage

load_dotenv('.env')
FIGMA_TOKEN = os.getenv('FIGMA_TOKEN')
FIGMA_DOCUMENT_ID1 = os.getenv('FIGMA_DOCUMENT_ID1')
FIGMA_DOCUMENT_ID2 = os.getenv('FIGMA_DOCUMENT_ID2')

page_list1=["Module 1","Module 2", "Module 3","Module 4"]
page_list2=["Module 5", "Module 6","Module 7","Module 8"]
os.environ["FIGMA_API_KEY"] = FIGMA_TOKEN

In [2]:
PHOTOBANK_CSV_PATH = "../02_Inputs/data/photobank.csv"

def load_photobank_images(csv_path=PHOTOBANK_CSV_PATH):
    images = []
    with open(csv_path, 'r', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for row in reader:
            images.append(row['file_path'])
    return images

PHOTOBANK_IMAGES = load_photobank_images()

In [3]:
##Generate english module structure

import pandas as pd
import json
import re
from pathlib import Path

# Utility functions
def normalize(text):
    return re.sub(r"[\u202f\xa0]", " ", str(text)).strip()

def clean_title(text, prefix=None):
    text = normalize(text)
    return re.sub(prefix, "", text).strip() if prefix else text

# Load data
lessons = pd.read_csv("../02_Inputs/data/sea-lessons.csv")
metadata = pd.read_csv("../02_Inputs/data/Module-Chapter-Details.csv").fillna("")

# Preprocess lesson data
lessons = lessons[lessons["Lesson Number"].notna()]
lessons["Lesson Number"] = lessons["Lesson Number"].astype(str)
lessons["Module Number"] = lessons["Lesson Number"].apply(lambda x: x.split(".")[0])
lessons["Chapter Number"] = lessons["Lesson Number"].apply(lambda x: ".".join(x.split(".")[:2]))
lessons["Module Title"] = lessons["Module Title"].ffill()
lessons["Chapter Title"] = lessons["Chapter Title"].ffill()

# Build lookup tables
module_meta = (
    metadata[metadata["type"] == "module"]
    .set_index("module_id")
    .rename_axis(None)  # optional: cleaner index
)
module_meta.index = module_meta.index.astype(str)
module_meta = module_meta.to_dict("index")

chapter_meta = (
    metadata[metadata["type"] == "chapter"]
    .set_index("chapter_id")
    .rename_axis(None)
)
chapter_meta.index = chapter_meta.index.astype(str)
chapter_meta = chapter_meta.to_dict("index")
lesson_desc = lessons.groupby("Lesson Number")["Description"].first().fillna("").to_dict()


# Generate module structure
structure = {"modules": []}


for module_id, mdf in lessons.groupby("Module Number"):
    module_id = str(module_id)
    meta = module_meta.get(module_id, {})
    module_title = clean_title(mdf["Module Title"].iloc[0], r"^Module\s*\d+:?\s*")

    module = {
        "id": module_id,
        "title": module_title,
        "color": meta.get("color", ""),
        "icon": f"https://ik.imagekit.io/seacademy/Icons/Modules/module{module_id}.svg",
        "image": {
            "src": f"https://ik.imagekit.io/seacademy/HeaderImages/Modules/module{module_id}.png",
            "caption": ""
        },
        "description": meta.get("description", ""),
        "percentComplete": "0",
        "estimatedTime": meta.get("estimatedTime", ""),
        "chapters": []
    }

    # Module intro (add preface only for Module 1)
    intro_lessons = []
    
    if module_id == "1":
        intro_lessons.append({
            "type": "lesson",
            "title": f"Module {module_id} Preface",
            "id": f"{module_id}.0.-1",
            "progress": "not_started",
            "description": "Preface to the module and overview of what’s ahead.",
            "image": {
                "src": f"https://ik.imagekit.io/seacademy/HeaderImages/Modules/module{module_id}.png",
                "caption": ""
            }
        })
    
    intro_lessons.append({
        "type": "lesson",
        "title": f"Module {module_id} Introduction",
        "id": f"{module_id}.0.0",
        "progress": "not_started"
    })
    
    module["chapters"].append({
        "id": f"{module_id}.0",
        "title": f"Module {module_id} Introduction",
        "icon": module["icon"],
        "lessons": intro_lessons
    })


    for i, (chapter_id, cdf) in enumerate(mdf.groupby("Chapter Number")):
        chapter_id = str(chapter_id)
        chapter_meta_entry = chapter_meta.get(chapter_id, {})
        chapter_title = clean_title(cdf["Chapter Title"].iloc[0], r"^Chapter\s*\d+:?\s*")

        chapter = {
            "id": chapter_id,
            "title": f"Chapter {i+1}: <strong>{chapter_title}</strong>",
            "icon": f"https://ik.imagekit.io/seacademy/Icons/Chapters/chapter{chapter_id}.svg",
            "image": {
                "src": f"https://ik.imagekit.io/seacademy/HeaderImages/Chapters/chapter{chapter_id}.png",
                "caption": ""
            },
            "description": chapter_meta_entry.get("description", ""),
            "estimatedTime": chapter_meta_entry.get("estimatedTime", ""),
            "lessons": []
        }

        # Chapter intro
        chapter["lessons"].append({
            "type": "lesson",
            "title": f"Chapter {i+1} Introduction",
            "id": f"{chapter_id}.0",
            "progress": "not_started"
        })


        # Real lessons
        for _, row in cdf.drop_duplicates(subset="Lesson Number").iterrows():
            lesson_id = row["Lesson Number"]
            if pd.isna(row["Lesson Title"]) or lesson_id not in lesson_desc:
                continue  # skip if no valid title or no metadata
        
            lesson_title = f"Lesson {lesson_id[-1]}: <strong>{clean_title(row['Lesson Title'])}</strong>"
            chapter["lessons"].append({
                "type": "lesson",
                "id": lesson_id,
                "title": lesson_title,
                "progress": "not_started",
                "description": lesson_desc.get(lesson_id, ""),
                "frame": {
                    "src": f"https://sehseadata.blob.core.windows.net/images/Lessons/frame/lesson{lesson_id.replace('.', '')}_frame.webp",
                    "caption": ""
                },
                "image": {
                    "src": f"https://sehseadata.blob.core.windows.net/images/Lessons/image/lesson{lesson_id.replace('.', '')}_image.webp",
                    "caption": ""
                }
            })


        # Chapter outro
        chapter["lessons"].append({
            "type": "lesson",
            "title": f"Chapter {i+1} Outro",
            "id": f"{chapter_id}.-1",
            "progress": "not_started",
            "description": "Outro of the chapter and wrap-up.",
            "image": {
                "src": f"https://ik.imagekit.io/seacademy/HeaderImages/Chapters/chapter{chapter_id}.png",
                "caption": ""
            }
        })

        # 4) Build tooltip now that all lessons are in place
        base_desc = chapter_meta_entry.get("description", "")
        
        lesson_lines = []
        for entry in chapter["lessons"]:
            lid = entry["id"]
            # Skip intro (ends with ".0") and outro (ends with ".-1")
            if lid.endswith(".0") or lid.endswith(".-1"):
                continue
            # Build a line like: "Lesson 1: <strong>Title</strong>."
            idx = len(lesson_lines) + 1
            lesson_lines.append(f"{entry['title']}.")
        
        if lesson_lines:
            # Join each line with a "<br>" prefix
            formatted_lessons = "<br> ".join(lesson_lines)
            chapter["tooltip"] = f"{base_desc} <br> {formatted_lessons}"
        else:
            chapter["tooltip"] = base_desc


        module["chapters"].append(chapter)
    
    # Module-level quiz lesson (e.g., 1.-1.0)
    quiz_lesson = {
        "type": "lesson",
        "title": f"Module {module_id} Quiz",
        "id": f"{module_id}.-1.0",
        "progress": "not_started",
        "description": "Test your understanding of all lessons in this module.",
        "image": {
            "src": f"https://ik.imagekit.io/seacademy/HeaderImages/Modules/module{module_id}.png",
            "caption": ""
        }
    }
    
    # Module outro chapter
    module_outro_chapter = {
        "id": f"{module_id}.-1",
        "title": f"Module {module_id} Outro",
        "icon": "https://ik.imagekit.io/seacademy/Icons/module-outro.svg",
        "description": meta.get("description", ""),
        "lessons": [
            quiz_lesson,
            {
                "type": "lesson",
                "title": f"Module {module_id} Outro",
                "id": f"{module_id}.-1.-1",
                "progress": "not_started"
            }
        ]
    }
    
    module["chapters"].append(module_outro_chapter)

    structure["modules"].append(module)

# Save JSON
with open("../03_Outputs/SEA_Modules/en/module_structure.json", "w", encoding="utf-8") as f:
    json.dump(structure, f, indent=2, ensure_ascii=False)

print("✅ module_structure.json generated successfully.")


✅ module_structure.json generated successfully.


In [4]:
clearedInfographicsList=['M1_C1_3','M5_C3_4', 'M4_C1_6', 'M3_C2_2', 'M8_C2_5', 'M5_C1_1', 'M3_C2_5', 'M5_C1_2', 'M1_C1_8', 'M3_C2_4', 'M8_C4_1', 'M8_C1_2', 'M3_C1_1', 'M2_C1_1', 'M8_C2_10', 'M4_C2_7', 'M8_C1_1', 'M5_C2_5', 'M5_C4_3', 'M3_C1_3', 'M2_C1_3', 'M7_C1_9', 'M4_C2_4', 'M8_C1_4', 'M3_C3_5', 'M3_C1_7', 'M8_C2_16', 'M4_C2_1', 'M4_C2_3', 'M5_C2_3', 'M3_C3_6', 'M2_C3_6', 'M5_C4_5', 'M3_C1_4', 'M2_C1_4', 'M2_C1_5', 'M3_C1_5', 'M8_C2_15', 'M2_C3_7', 'M7_C1_10', 'M7_C1_2', 'M2_C1_9', 'M5_C4_8', 'M7_C3_1', 'M6_C3_1', 'M2_C1_13', 'M6_C1_3', 'M1_C2_4', 'M8_C1_8', 'M2_C1_11', 'M7_C3_3', 'M6_C3_2', 'M2_C3_8', 'M3_C3_8', 'M8_C1_9', 'M1_C2_2', 'M6_C3_5', 'M7_C3_4', 'M6_C1_6', 'M1_C2_1', 'M7_C2_3']

In [5]:


# ===============================
# Constants & Global Variables
# ===============================
FIGMA_FOLDER = "../02_Inputs/figma_jsons"
OUTPUT_DIR_BASE = "../03_Outputs/SEA_Modules/en"

LESSON_NEXT_MAP = {}


# ===============================
# Figma Document Caching Helpers
# ===============================
def save_figma_to_json(figma_document, filename):
    os.makedirs(FIGMA_FOLDER, exist_ok=True)
    with open(os.path.join(FIGMA_FOLDER, filename), 'w', encoding='utf-8') as f:
        json.dump(figma_document.model_dump(), f, indent=2)

def load_cached_figma_page(filename, target_page_name):
    """
    Loads a cached Figma document JSON file and returns the page that matches target_page_name.
    """
    file_path = os.path.join(FIGMA_FOLDER, filename)
    if not Path(file_path).exists():
        raise ValueError(f"No cached Figma document found in {file_path}. Set redownload=True to download.")
    print("Loading cached Figma document...")
    with open(file_path, 'r', encoding='utf-8') as f:
        figma_data = json.load(f)
    figma_document = Document.model_validate(figma_data)
    try:
        page = next(
            page for page in figma_document.children
            if page.type == "CANVAS" and page.name == target_page_name
        )
        print(f"Found page: {target_page_name}")
        return page
    except StopIteration:
        raise ValueError(f"Page '{target_page_name}' not found in the document.")


# ===============================
# Section & Lesson Extraction
# ===============================
def get_valid_sections(page):
    """
    Filters the sections on a given page returning only those matching allowed section names.
    """
    print("Filtering relevant sections...")
    allowed_sections = {"Module 1","Module 2","Module 3","Module 4","Module 5","Module 6", "Module 7","Module 8","Module Intro","Chapter 1", "Chapter 2", "Chapter 3", "Chapter 4", "Chapter 5", "Module Outro"}
    sections = [section for section in page.children if section.type == "SECTION" and section.name in allowed_sections]
    print(f"Found {len(sections)} relevant sections out of {len(page.children)}")
    return sections

def extract_lessons(valid_sections, module_number):
    """
    Groups frames into lessons based on section/subsection names.
    Lessons are identified using a chapter and lesson numbering scheme.
    """
    lesson_groups = defaultdict(list)
    lesson_section_pattern = re.compile(r"Lesson (\d+)", re.IGNORECASE)
    
    for section in valid_sections:
        print(f"\n➡️ Processing section: {section.name}")
        section_name_lower = section.name.lower()
        
        # Handle module intro/outro sections differently
        if section_name_lower in ["module intro", "module outro"]:
            chapter_num = -1 if "outro" in section_name_lower else 0
            lesson_num = 0 if "intro" in section_name_lower else -1
            lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
            print(f"  🔹 Found lesson: {lesson_id}")
            for frame in section.children:
                if frame.type == "FRAME":
                    lesson_groups[lesson_id].append(frame)
            continue

        if section_name_lower in ["module 1"]:
            for subsection in section.children:
                sub_name = subsection.name.lower()
                chapter_num = 0
                lesson_num = -1 if "preface" in sub_name else 0
                lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
                print(f"  🔹 Found lesson: {lesson_id}")
                for frame in subsection.children:
                    if frame.type == "FRAME":
                        lesson_groups[lesson_id].append(frame)
                continue
            
        # Determine the chapter number from section
        chapter_match = re.search(r"Chapter (\d+)", section.name)
        if chapter_match:
            chapter_num = int(chapter_match.group(1))
        elif section.name.lower() == "module intro":
            chapter_num = 0
        elif section.name.lower() == "module outro" or section.name.lower() == "module preface":
            chapter_num = -1
        else:
            chapter_num = -99
        
        # Process subsections (lesson-specific) within each section
        for subsection in section.children:
            if subsection.type != "SECTION":
                continue
            sub_name = subsection.name.lower()
            if "chapter intro" in sub_name:
                lesson_num = 0
            elif "chapter outro" in sub_name:
                lesson_num = -1
            elif lesson_section_pattern.match(subsection.name):
                lesson_num = int(lesson_section_pattern.match(subsection.name).group(1))
            else:
                continue
            lesson_id = f"{module_number}.{chapter_num}.{lesson_num}"
            print(f"  🔹 Found lesson: {lesson_id}")
            for frame in subsection.children:
                if frame.type == "FRAME":
                    lesson_groups[lesson_id].append(frame)
    return lesson_groups


# ===============================
# Frame Processing
# ===============================
def build_frame_class_map(raw_map):
    """
    Given a dictionary of raw class mappings, attempts to evaluate each class.
    Returns a mapping of frame names to class objects and a set of names that could not be evaluated.
    """
    frame_map = {}
    skipped = set()
    for name, class_name in raw_map.items():
        try:
            frame_map[name] = eval(class_name)
        except NameError:
            skipped.add(name)
    return frame_map, skipped

def get_color_scheme(frame):
    """
    Infer the color_scheme ("dark" or "light") based on the frame background color.
    """
    if hasattr(frame, "backgroundColor") and frame.backgroundColor:
        color = frame.backgroundColor
        r, g, b = color.get("r", 0), color.get("g", 0), color.get("b", 0)
        brightness = 0.2126 * r + 0.7152 * g + 0.0722 * b
        return "dark" if brightness < 0.5 else "light"
    return "unknown"

def process_frame(frame, idx, frame_class_map, skipped_set, missing_templates, errors):
    """
    Process an individual frame using its mapped class.
    If the frame name is not in the mapping or processing fails, record the issue and return None.
    """
    if frame.name not in frame_class_map:
        if frame.name not in skipped_set and not frame.name.lower().endswith("_ignore"):
            missing_templates.add(frame.name)
        return None  # or use create_fallback(frame, idx)
    cls = frame_class_map[frame.name]
    try:
        segment = cls.from_node(frame).to_content()
        segment["_order_index"] = idx
        segment["color_scheme"] = get_color_scheme(frame)
        return segment
    except Exception as e:
        print(e)
        key = (frame.name, cls.__name__, str(e).strip().split("\n")[0])
        errors.add(key)
        return None  # or use create_fallback(frame, idx, str(e))

def process_lesson(lesson_id, frame_list, frame_class_map, skipped_set, errors, missing_templates):
    """
    Processes all frames for one lesson.
    Reverses the frame list to account for Figma’s visual ordering,
    then sorts the segments by an internal order index.
    """
    segments = []
    reversed_frames = list(reversed(frame_list))
    for idx, frame in enumerate(reversed_frames):
        segment = process_frame(frame, idx, frame_class_map, skipped_set, missing_templates, errors)
        if segment is not None:
            segments.append(segment)
    segments.sort(key=lambda s: s["_order_index"])
    for s in segments:
        s.pop("_order_index", None)
    return segments


# ===============================
# Unicode Cleaning & File Output
# ===============================
def clean_unicode(text):
    """
    Remove control characters (except newline and tab) and normalize whitespace.
    """
    if not isinstance(text, str):
        return text
    cleaned = ''.join(
        c if unicodedata.category(c)[0] != 'C' or c in '\n\t' else ' '
        for c in text
    )
    return ' '.join(cleaned.split())

def recursively_clean(data):
    """
    Recursively cleans all string values within nested dicts and lists.
    """
    if isinstance(data, dict):
        return {k: recursively_clean(v) for k, v in data.items()}
    elif isinstance(data, list):
        return [recursively_clean(v) for v in data]
    elif isinstance(data, str):
        return clean_unicode(data)
    else:
        return data

def write_lesson_output(lesson_id, segments, output_dir):
    """
    Write the cleaned lesson output to a JSON file.
    """
    cleaned_segments = recursively_clean(segments)
    lesson_output = {"id": lesson_id, "segments": cleaned_segments}
    filename = f"{lesson_id}.json"
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, filename), "w", encoding="utf-8") as f:
        json.dump(lesson_output, f, indent=2, sort_keys=False, ensure_ascii=False)
    print(f"✅ Exported lesson {lesson_id}")


# ===============================
# Photobank & Next-Lesson Mapping (this section updated has new content)
# ===============================


def build_lesson_next_map(module_structure_path="../03_Outputs/SEA_Modules/en/module_structure.json"):
    """
    Extracts lesson IDs in order from module_structure.json and creates a mapping
    from each lesson id to the next lesson id.
    """
    global LESSON_NEXT_MAP
    with open(module_structure_path, "r", encoding="utf-8") as f:
        structure = json.load(f)
    lessons = []
    for module in structure.get("modules", []):
        for chapter in module.get("chapters", []):
            for lesson in chapter.get("lessons", []):
                lesson_id = lesson.get("id")
                if lesson_id:
                    lessons.append(lesson_id)
    lesson_map = {lesson_id: lessons[i+1] if i < len(lessons) - 1 else lesson_id 
                  for i, lesson_id in enumerate(lessons)}
    LESSON_NEXT_MAP = lesson_map
    return LESSON_NEXT_MAP

# def post_process_images(segments,lesson_id):
#     root_file_path = "https://sehseadata.blob.core.windows.net/images/Modules"#"https://ik.imagekit.io/seacademy/Photos"
#     module_num = lesson_id.split(".")[0]
#     module_name = f"module_{module_num}".capitalize()
#     def update_image(image):
#         if "src" in image:
#             # if " " in str(original):
#             #     original=original.strip
#             # if module_num in ["5","1"] or original in clearedInfographicsList:
#             #     #print(original)
#             try:
#                 original = image["src"].replace(":","_").strip()
#                 image["src"] = f"{root_file_path}/{module_name}/{original}.webp"
#             except:
#                 random_image = random.choice(PHOTOBANK_IMAGES) if PHOTOBANK_IMAGES else original
#                 image["src"] = f"{root_file_path}/{random_image}"
#                 image["caption"]=""

def post_process_images(segments, lesson_id):
    root_file_path = "https://sehseadata.blob.core.windows.net/images/Modules"
    module_num = lesson_id.split(".")[0]
    module_name = f"module_{module_num}".capitalize()
    
    def update_image(image):
        if "src" in image:
            try:
                original = image["src"].replace(":", "_").strip()
                # If module is 4, 7, or 8, override with random photo from photobank
                if module_num in ["4", "7", "8"]:
                    random_image = random.choice(PHOTOBANK_IMAGES) if PHOTOBANK_IMAGES else original
                    image["src"] = f"{root_file_path}/{random_image}"
                    image["caption"] = ""
                else:
                    image["src"] = f"{root_file_path}/{module_name}/{original}.webp"
            except:
                # Fallback in case of an error
                image["src"] = f"{root_file_path}/{random.choice(PHOTOBANK_IMAGES)}"
                image["caption"] = ""
    
    def update_data(data):
        if isinstance(data, dict):
            if "image" in data and isinstance(data["image"], dict):
                update_image(data["image"])
            for value in data.values():
                update_data(value)
        elif isinstance(data, list):
            for item in data:
                update_data(item)
    
    for seg in segments:
        update_data(seg)
    return segments
# ===============================
# Refactor Global Maps Loader: Only Use 2 Files
# ===============================

import pandas as pd
import json
from collections import defaultdict
import re

def strip_html_tags(text):
    return re.sub(r'<[^>]*>', '', text or "")

def build_lesson_metadata_only(module_json_path):
    """
    Rebuilds lesson metadata using only module_structure.json
    Returns: LESSON_METADATA, CONCEPTS_MAP, RESOURCES_MAP
    """
    with open(module_json_path, "r", encoding="utf-8") as f:
        structure = json.load(f)

    lesson_meta = {}
    for module in structure.get("modules", []):
        for chapter in module.get("chapters", []):
            chapter_id = chapter.get("id")
            chapter_image = chapter.get("image", {})
            chapter_desc = strip_html_tags(chapter.get("description", ""))
            for lesson in chapter.get("lessons", []):
                lesson_id = lesson.get("id")
                if lesson_id:
                    lesson_meta[lesson_id] = {
                        "title": strip_html_tags(lesson.get("title", "")),
                        "image": lesson.get("image", {}).get("src") or chapter_image.get("src", ""),
                        "frame": lesson.get("frame", {}).get("src") or chapter_image.get("src", ""),
                        "caption": lesson.get("image", {}).get("caption") or chapter_image.get("caption", ""),
                        "description": lesson.get("description") or chapter_desc,
                        "progress": lesson.get("progress", "not_started")
                    }

    # Load csv if available to build CONCEPTS_MAP and RESOURCES_MAP
    concepts_map = defaultdict(list)
    resources_map = defaultdict(list)

    try:
        lessons_df = pd.read_csv("../02_Inputs/data/sea-lessons.csv")
        for _, row in lessons_df.iterrows():
            lesson_id = str(row.get("Lesson Number", "")).strip()
            key_concept = row.get("Key Concept", "")
            concept_def = row.get("Concept Definition", "")
        
            # Replace NaN with empty strings
            if pd.isna(key_concept):
                key_concept = ""
            if pd.isna(concept_def):
                concept_def = ""
        
            concept = {
                "title": key_concept,
                "body": concept_def,
                "source": ""
            }
            concepts_map[lesson_id].append(concept)

            # Build resource image src like: https://ik.imagekit.io/seacademy/Resources/Module_1/resource-3211.png
            module_num = lesson_id.split(".")[0]
            numeric_id = lesson_id.replace(".", "")+str(row["Resource Number"])
            resource_img = f"https://ik.imagekit.io/seacademy/Resources/Module_{module_num}/resource-{numeric_id}.png"

            resource = {
                "image": {
                    "src": resource_img,
                    "caption": ""
                },
                "href": row["Resource Link"],
                "text": row["Resource Title"],
                "cta": "Click to download"
            }
            resources_map[lesson_id].append(resource)
    except Exception as e:
        print(f"⚠️ Could not load concepts/resources from csv: {e}")

    return lesson_meta, dict(concepts_map), dict(resources_map)

def build_lesson_next_map(module_structure_path):
    """
    Extracts lesson IDs in order from module_structure.json and creates a mapping
    from each lesson id to the next lesson id.
    """
    with open(module_structure_path, "r", encoding="utf-8") as f:
        structure = json.load(f)
    lessons = []
    for module in structure.get("modules", []):
        for chapter in module.get("chapters", []):
            for lesson in chapter.get("lessons", []):
                lesson_id = lesson.get("id")
                if lesson_id:
                    lessons.append(lesson_id)
    lesson_map = {lesson_id: lessons[i+1] if i < len(lessons) - 1 else lesson_id 
                  for i, lesson_id in enumerate(lessons)}
    return lesson_map


import pandas as pd

def build_quiz_object(module_number: int) -> dict:
    """
    Builds a quiz segment object for a given module by reading the corresponding sheet in SEA Quizzes.xlsx.
    Supports multiple correct answers in the form '1,2,4'.
    """
    path = "../02_Inputs/SEA Quizzes.xlsx"
    try:
        df = pd.read_excel(path, sheet_name=f"Module {module_number}")
    except Exception as e:
        print(f"⚠️ Quiz sheet for Module {module_number} not found: {e}")
        return {}

    df = df.dropna(subset=["Question Text"])

    questions = []
    for i, row in df.iterrows():
        # Prepare answer options
        options = [
            {"id": j, "value": str(row[col]).strip()}
            for j, col in enumerate(["Option 1", "Option 2", "Option 3", "Option 4"], start=1)
            if pd.notna(row[col])
        ]

        # Parse correct answer(s)
        raw_correct = str(row.get("Correct", "")).strip()
        try:
            correct_answers = [int(x) for x in raw_correct.split(",") if x.strip().isdigit()]
        except:
            print(f"⚠️ Invalid correct answer format for question {i + 1} in Module {module_number}: {raw_correct}")
            continue

        if not correct_answers:
            continue

        question = {
            "id": i + 1,
            "prompt": str(row["Question Text"]).strip(),
            "multiple": len(correct_answers) > 1,
            "options": options,
            "solution": correct_answers,
            "messages": {
                "correct": {
                    "title": "Correct",
                    "body": str(row.get("Answer Text", "")).strip()
                },
                "wrong": {
                    "title": "Incorrect",
                    "body": str(row.get("Answer Text", "")).strip()
                }
            }
        }
        questions.append(question)

    if not questions:
        return {}

    return {
        "template_id": "scored_quiz",
        "color_scheme": "dark",
        "content": {
            "title": f"Scored quiz <br/><strong>Module {module_number}</strong>",
            "intro": "This quiz covers all the lessons in this module. You must achieve 80% to pass. Good luck!",
            "final": False,
            "labels": {
                "correct": "Correct",
                "question": "Question",
                "result": "Quiz Result",
                "score": "Your score",
                "passingScore": "Minimum passing score",
                "startButton": "Start Quiz",
                "nextButton": "Next",
                "submitButton": "Submit",
                "retryButton": "Replay",
                "passed": "Passed",
                "failed": "Not passed"
            },
            "passingScore": 0.8,
            "questions": questions
        }
    }



def add_list_of_lessons(segments, lesson_id, lesson_metadata):
    """
    Appends a 'list_of_lessons' segment to chapter intro lessons (e.g. '1.1.0').
    Excludes module intro/outro (e.g. '1.0.0', '1.-1.0').
    """
    chapter_intro_pattern = re.compile(r'^(\d+)\.(\d+)\.0$')
    match = chapter_intro_pattern.match(lesson_id)
    if not match:
        return segments  # Not a chapter intro

    module_num, chapter_num = match.groups()
    if chapter_num in ["0", "-1"]:
        return segments  # Skip module intro/outro

    list_items = []
    for lid, data in lesson_metadata.items():
        if not lid.startswith(f"{module_num}.{chapter_num}."):
            continue
        if lid == lesson_id or lid.endswith(".0") or lid.endswith(".-1"):
            continue  # skip intro/outro and self
        list_items.append({
            "title": data.get("title", ""),
            "lessonId": lid,
            "type": "lesson",
            "description": data.get("description", ""),
            "progress": data.get("progress", "not_started"),
            "cta": "Go to the lesson",
            "image": {"src":data.get("frame", ""),
                      "caption":data.get("caption","")}
        })

    if list_items:
        list_of_lessons_segment = {
            "template_id": "list_of_lessons",
            "color_scheme": "dark",
            "content": {
                "title": "What's next in this chapter?",
                "lessons": list_items
            }
        }
        segments.append(list_of_lessons_segment)

    return segments

# Example Usage (Replace path with actual if needed):
MODULE_STRUCTURE_PATH = "../03_Outputs/SEA_Modules/en/module_structure.json"
LESSON_METADATA, CONCEPTS_MAP, RESOURCES_MAP = build_lesson_metadata_only(MODULE_STRUCTURE_PATH)
LESSON_NEXT_MAP = build_lesson_next_map(MODULE_STRUCTURE_PATH)

def add_post_segments(segments, lesson_id):

    if segments is None:
        print(f"⚠️ Warning: segments for lesson {lesson_id} is None, skipping post-processing.")
        return []
    """
    Adds 3 segments: key_concepts (3rd), key_resources (2nd to last), and connection_next (last)
    based on global maps: CONCEPTS_MAP, RESOURCES_MAP, LESSON_METADATA, LESSON_NEXT_MAP
    """
    if not LESSON_NEXT_MAP:
        build_lesson_next_map()

    root_file_path = "https://ik.imagekit.io/seacademy/Photos"
    module_num = lesson_id.split(".")[0]
    module_name = f"module_{module_num}".capitalize()

    
    def update_image_path(image):
 

        if "src" in image and not image["src"].startswith("http"):
            image["src"] = f"{root_file_path}/{module_name}/{image['src']}"
            image["caption"]=""

    if lesson_id[-1] != "0" and lesson_id[-2:]!="-1":
    
        # --- Insert key_concepts (3rd segment) ---
        concept_items = CONCEPTS_MAP.get(lesson_id, [])[:3]
        key_concepts = {
            "template_id": "key_concepts",
            "color_scheme": "dark",
            "content": {
                "title": "Key Concepts",
                "intro": "Explore foundational ideas from this lesson.",
                "concepts": concept_items
            }
        }
        segments.insert(2, key_concepts)
    
        # --- Insert key_resources (second to last) ---
        resource_items = RESOURCES_MAP.get(lesson_id, [])[:3]
        for r in resource_items:
            update_image_path(r["image"])
        key_resources = {
            "template_id": "key_resources",
            "color_scheme": "dark",
            "content": {
                "title": "Key Resources",
                "resources": resource_items
            }
        }
        segments.append(key_resources)

    # --- Append connection_next (last) ---
    next_id = LESSON_NEXT_MAP.get(lesson_id, lesson_id)
    next_data = LESSON_METADATA.get(next_id, {})
    connection_next = {
        "template_id": "connection_next",
        "color_scheme": "dark",
        "content": {
            "image": {
                "src": next_data.get("image", "https://coolermed.com/wp-content/uploads/2023/02/what-is-sustainable-energy-how-to-apply-it-to-medical-use.jpg"),
                "caption": ""#next_data.get("caption", "image caption")
            },
            "intro": "Next up",
            "title": next_data.get("title", "Next Lesson"),
            "cta": "Start learning",
            "nextLessonId": next_id
        }
    }
    update_image_path(connection_next["content"]["image"])
    segments.append(connection_next)

    return segments



# ===============================
# Export & Summary Functions
# ===============================
def export_lessons(lesson_groups, frame_class_map, skipped_templates, output_base_dir):
    """
    Processes each lesson into segments, postprocesses them, and writes each as a JSON file.
    """
    os.makedirs(output_base_dir, exist_ok=True)
    skipped_set = set(skipped_templates)
    errors = set()
    missing_templates = set()
    for lesson_id, frame_list in lesson_groups.items():
        segments = process_lesson(lesson_id, frame_list, frame_class_map, skipped_set, errors, missing_templates)

        segments = post_process_images(segments,lesson_id)
        segments = add_list_of_lessons(segments, lesson_id, LESSON_METADATA)
        segments = add_post_segments(segments,lesson_id)
        write_lesson_output(lesson_id, segments, output_base_dir)

    # Infer module number from the output path (e.g. ".../Module_4")
    match = re.search(r"Module[_ ](\d+)", output_base_dir, re.IGNORECASE)
    module_number = int(match.group(1)) if match else None
    # Add module-level quiz lesson (e.g., 4.-1.0)
    if module_number is not None:
        quiz_segment = build_quiz_object(module_number)
        print("quiz",module_number,quiz_segment)
        if quiz_segment:
            quiz_lesson_id = f"{module_number}.-1.0"
            write_lesson_output(quiz_lesson_id, [quiz_segment], output_base_dir)
    
    return errors, missing_templates

def print_summary(errors, skipped_templates, missing_runtime_templates, output_dir, lesson_count):
    print(f"\nFinished. {lesson_count} lessons processed from filtered sections. Output in: {output_dir}")
    if errors:
        print(f"\n⚠️ Encountered {len(errors)} unique errors during frame parsing:")
        for frame_name, class_name, error_msg in sorted(errors):
            print(f" - [{class_name}] {frame_name}: {error_msg}")
    if skipped_templates:
        print("\n⚠️ Skipped templates due to missing class definitions (initial map):")
        for template in sorted(skipped_templates):
            print(f" - {template}")
    if missing_runtime_templates:
        print("\n⚠️ Skipped templates not found in class map (encountered during lessons):")
        for template in sorted(missing_runtime_templates):
            print(f" - {template}")


# ===============================
# Pipeline Orchestration
# ===============================
def process_pages(page_list, loader_func, raw_frame_class_map):
    """
    Iterates through a list of page names, loads each page using the provided loader function,
    extracts lessons and exports them.
    """
    for page_name in page_list:
        print(f"\n========================\n📄 Processing {page_name}\n========================")
        page = loader_func(page_name)
        valid_sections = get_valid_sections(page)
        frame_class_map, skipped_templates = build_frame_class_map(raw_frame_class_map)
        module_number_match = re.search(r"\d+", page_name)
        if not module_number_match:
            raise ValueError(f"Module number not found in page name: {page_name}")
        module_number = int(module_number_match.group())
        lesson_groups = extract_lessons(valid_sections, module_number)
        print(f"\nGrouped into {len(lesson_groups)} lessons.")
        output_dir = os.path.join(OUTPUT_DIR_BASE, page_name.replace(" ", "_"))
        errors, missing_runtime_templates = export_lessons(lesson_groups, frame_class_map, skipped_templates, output_dir)
        print_summary(errors, skipped_templates, missing_runtime_templates, output_dir, len(lesson_groups))

def run_pipeline(page_list1, page_list2, redownload=False):
    """
    Main pipeline function that optionally re-downloads the Figma documents and processes
    pages from both document versions.
    """
    if redownload:
        print("Downloading Figma document1...")
        FIGMA_DOCUMENT1 = Document.from_file_key(FIGMA_DOCUMENT_ID1)
        print("Downloading Figma document2...")
        FIGMA_DOCUMENT2 = Document.from_file_key(FIGMA_DOCUMENT_ID2)
        save_figma_to_json(FIGMA_DOCUMENT1, "figma_document1.json")
        save_figma_to_json(FIGMA_DOCUMENT2, "figma_document2.json")
    
    # Note: The commented class map entries below are preserved for later re-enabling:
    raw_frame_class_map = {
        "lesson_cover": "LessonCover",
        "module_cover": "ModuleCover",
        "learning_objectives": "LearningObjectives",
        "photo-vertical": "PhotoVertical",
        "photo-horizontal": "PhotoHorizontal",
        "photo-full-height": "PhotoFullHeight",
        "text": "ModuleText",  
        "lesson_subpart_cover": "LessonSubpartCover",
        "lesson_part_cover": "LessonPartCover",
        "chart":"Chart",
        "infographic":"Infographic",
        #"chapter_outro": "ChapterOutro",
        "module_outro": "ModuleOutro",
        "key_takeaways": "KeyTakeaways",    
        "chapter_cover": "ChapterCover" 
    }
    
    process_pages(page_list1, lambda name: load_cached_figma_page("figma_document1.json", name), raw_frame_class_map)
    process_pages(page_list2, lambda name: load_cached_figma_page("figma_document2.json", name), raw_frame_class_map)


# Example usage:
run_pipeline(page_list1, page_list2, redownload=False)



📄 Processing Module 1
Loading cached Figma document...
Found page: Module 1
Filtering relevant sections...
Found 6 relevant sections out of 7

➡️ Processing section: Module Outro
  🔹 Found lesson: 1.-1.-1

➡️ Processing section: Chapter 4
  🔹 Found lesson: 1.4.-1
  🔹 Found lesson: 1.4.3
  🔹 Found lesson: 1.4.2
  🔹 Found lesson: 1.4.1
  🔹 Found lesson: 1.4.0

➡️ Processing section: Chapter 3
  🔹 Found lesson: 1.3.-1
  🔹 Found lesson: 1.3.5
  🔹 Found lesson: 1.3.4
  🔹 Found lesson: 1.3.3
  🔹 Found lesson: 1.3.2
  🔹 Found lesson: 1.3.1
  🔹 Found lesson: 1.3.0

➡️ Processing section: Chapter 2
  🔹 Found lesson: 1.2.-1
  🔹 Found lesson: 1.2.3
  🔹 Found lesson: 1.2.2
  🔹 Found lesson: 1.2.1
  🔹 Found lesson: 1.2.0

➡️ Processing section: Chapter 1
  🔹 Found lesson: 1.1.-1
  🔹 Found lesson: 1.1.3
  🔹 Found lesson: 1.1.2
  🔹 Found lesson: 1.1.1
  🔹 Found lesson: 1.1.0

➡️ Processing section: Module 1
  🔹 Found lesson: 1.0.0
  🔹 Found lesson: 1.0.-1

Grouped into 25 lessons.
'NoneType' object 

In [6]:

# ##fake module 8 data

# import os
# import json

# def extract_cover_and_next_segments(module_structure_path, module_number, output_dir):
#     def get_cover_type(lesson_id):
#         m, c, l = lesson_id.split(".")
#         if l == "0":
#             return "module_cover" if c == "0" else "chapter_cover"
#         if l.startswith("-"):
#             return None  # no cover for chapter/module outros
#         return "lesson_cover"

#     with open(module_structure_path, "r", encoding="utf-8") as f:
#         structure = json.load(f)

#     output_dir = os.path.join(output_dir, f"Module_{module_number}")
#     os.makedirs(output_dir, exist_ok=True)

#     for module in structure.get("modules", []):
#         if str(module["id"]) != str(module_number):
#             continue

#         module_title = module.get("title", f"Module {module_number}")
#         module_image = module.get("image", {})

#         all_lessons = []
#         lesson_to_chapter = {}

#         for chapter in module.get("chapters", []):
#             chapter_id = chapter.get("id", "")
#             chapter_number = chapter_id.split(".")[1] if "." in chapter_id else "X"
#             chapter_title = chapter.get("title", f"Chapter {chapter_number}")
#             chapter_image = chapter.get("image", {})
#             for lesson in chapter.get("lessons", []):
#                 lesson_id = lesson.get("id")
#                 if lesson_id:
#                     all_lessons.append({
#                         "lesson": lesson,
#                         "chapter_number": chapter_number,
#                         "chapter_title": chapter_title,
#                         "chapter_image": chapter_image
#                     })
#                     lesson_to_chapter[lesson_id] = chapter

#         for i, item in enumerate(all_lessons):
#             lesson = item["lesson"]
#             chapter_number = item["chapter_number"]
#             chapter_title = item["chapter_title"]
#             chapter_image = item["chapter_image"]

#             lesson_id = lesson["id"]
#             lesson_title = lesson.get("title", "")
#             segments = []

#             cover_type = get_cover_type(lesson_id)
#             if cover_type == "module_cover":
#                 segments.append({
#                     "template_id": "module_cover",
#                     "color_scheme": "light",
#                     "content": {
#                         "image": {
#                             "src": module_image.get("src", ""),
#                             "caption": module_image.get("caption", "")
#                         },
#                         "module": {
#                             "label": "Module",
#                             "number": str(module_number)
#                         },
#                         "title": module_title,
#                         "cta": "Scroll, tab or use your keyboard to move ahead"
#                     }
#                 })
#             elif cover_type == "chapter_cover":
#                 segments.append({
#                     "template_id": "chapter_cover",
#                     "color_scheme": "dark",
#                     "content": {
#                         "image": {
#                             "src": chapter_image.get("src", ""),
#                             "caption": chapter_image.get("caption", "")
#                         },
#                         "chapter": {
#                             "label": "Chapter",
#                             "number": str(chapter_number)
#                         },
#                         "title": chapter_title,
#                         "cta": "Scroll, tab or use your keyboard to move ahead",
#                         "intro": f"M{module_number}: {module_title} | Chapter {chapter_number}"
#                     }
#                 })
#             elif cover_type == "lesson_cover":
#                 lesson_number = lesson_id.split(".")[2]
#                 segments.append({
#                     "template_id": "lesson_cover",
#                     "color_scheme": "dark",
#                     "content": {
#                         "image": {
#                             "src": chapter_image.get("src", ""),
#                             "caption": chapter_image.get("caption", "")
#                         },
#                         "lesson": {
#                             "label": "Lesson",
#                             "number": str(lesson_number)
#                         },
#                         "title": lesson_title,
#                         "cta": None,
#                         "intro": f"M{module_number}: {module_title} | Chapter {chapter_number} | Lesson {lesson_number}"
#                     }
#                 })

#             # Add connection_next — always
#             if i + 1 < len(all_lessons):
#                 next_item = all_lessons[i + 1]
#                 next_lesson = next_item["lesson"]
                
#                 next_image = next_item["chapter_image"]
#                 segments.append({
#                     "template_id": "connection_next",
#                     "color_scheme": "dark",
#                     "content": {
#                         "image": {
#                             "src": next_image.get("src", ""),
#                             "caption": next_image.get("caption", "")
#                         },
#                         "intro": "Next up",
#                         "title": next_lesson.get("title", "Next Lesson"),
#                         "cta": "Start learning",
#                         "nextLessonId": next_lesson.get("id")
#                     }
#                 })

#             output = {
#                 "id": lesson_id,
#                 "segments": segments
#             }
#             with open(os.path.join(output_dir, f"{lesson_id}.json"), "w", encoding="utf-8") as f:
#                 json.dump(output, f, indent=2, ensure_ascii=False)

#     print(f"✅ Fully generated: Module {module_number} JSONs with correct covers + connection_next.")

# extract_cover_and_next_segments(
#     module_structure_path="../03_Outputs/SEA_Modules/en/module_structure.json",
#     module_number=8,
#     output_dir="../03_Outputs/SEA_Modules/en/"
# )


In [7]:
def collect_lesson_ids_from_folder(root_folder):
    """
    Walks through all subdirectories of root_folder, finds .json files,
    strips the '.json' extension, and returns the list.
    """
    lesson_ids = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.endswith(".json") and filename != "module_structure.json":
                lesson_id = filename[:-5]  # Remove ".json"
                lesson_ids.append(lesson_id)
    return lesson_ids

all_lessons = collect_lesson_ids_from_folder("../03_Outputs/SEA_Modules/en")
print("\n🗂️ All lesson IDs found:")
print(all_lessons)

open("../03_Outputs/all_lesson_ids.txt", "w").write(str(all_lessons))



🗂️ All lesson IDs found:
['4.5.2', '4.2.0', '4.4.-1', '4.2.1', '4.1.4', '4.5.3', '4.0.0', '4.5.-1', '4.4.0', '4.1.3', '4.-1.0', '4.3.2', '4.-1.-1', '4.3.3', '4.1.2', '4.4.1', '4.3.-1', '4.3.0', '4.1.1', '4.4.2', '4.4.3', '4.2.-1', '4.1.0', '4.3.1', '4.2.2', '4.5.0', '4.4.4', '4.5.1', '4.1.-1', '4.2.3', '3.3.3', '3.1.2', '3.4.1', '3.-1.-1', '3.4.0', '3.-1.0', '3.1.3', '3.3.2', '3.2.1', '3.0.0', '3.4.-1', '3.2.0', '3.2.3', '3.1.-1', '3.2.2', '3.4.3', '3.1.0', '3.3.1', '3.2.-1', '3.3.0', '3.1.1', '3.4.2', '3.3.-1', '2.3.3', '2.1.2', '2.1.3', '2.3.2', '2.3.5', '2.2.1', '2.0.0', '2.-1.0', '2.2.0', '2.3.4', '2.2.3', '2.2.-1', '2.3.-1', '2.2.2', '2.-1.-1', '2.1.-1', '2.1.0', '2.3.1', '2.3.0', '2.1.1', '5.-1.0', '5.2.0', '5.2.1', '5.0.0', '5.4.0', '5.1.3', '5.3.2', '5.4.-1', '5.3.3', '5.1.2', '5.4.1', '5.3.0', '5.1.1', '5.4.2', '5.1.-1', '5.4.3', '5.1.0', '5.3.1', '5.3.-1', '5.-1.-1', '5.2.2', '5.4.4', '5.2.3', '5.2.-1', '7.1.0', '7.3.1', '7.1.-1', '7.3.0', '7.1.1', '7.4.2', '7.3.-1', '7.2.3'

1776

In [8]:
import os
import json
import math


def find_invalid_numbers(obj, path="$"):
    """
    Recursively traverse a JSON-parsed object and collect paths to any float values
    that are NaN or infinite.
    """
    invalids = []
    if isinstance(obj, dict):
        for key, value in obj.items():
            invalids.extend(find_invalid_numbers(value, f"{path}.{key}"))
    elif isinstance(obj, list):
        for idx, value in enumerate(obj):
            invalids.extend(find_invalid_numbers(value, f"{path}[{idx}]") )
    elif isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            invalids.append((path, obj))
    return invalids


def sanitize_invalid_numbers(obj):
    """
    Recursively traverse the JSON object and replace NaN or infinite floats with empty strings.
    """
    if isinstance(obj, dict):
        return {k: sanitize_invalid_numbers(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [sanitize_invalid_numbers(v) for v in obj]
    if isinstance(obj, float):
        if math.isnan(obj) or math.isinf(obj):
            return ""
        return obj
    return obj


def validate_and_sanitize_json_files(root_folder):
    """
    Walks through all JSON files under root_folder; for any file containing NaN or infinite values,
    replaces those values with empty strings and rewrites the file.

    Returns a list of files that were sanitized and their original invalid entries.
    """
    sanitized_files = []
    for dirpath, _, filenames in os.walk(root_folder):
        for filename in filenames:
            if filename.lower().endswith('.json'):
                file_path = os.path.join(dirpath, filename)
                try:
                    with open(file_path, 'r', encoding='utf-8') as f:
                        data = json.load(f)
                    invalid_entries = find_invalid_numbers(data)
                    if invalid_entries:
                        msgs = "; ".join(f"{p}={v}" for p, v in invalid_entries)
                        sanitized_files.append((file_path, msgs))
                        cleaned = sanitize_invalid_numbers(data)
                        with open(file_path, 'w', encoding='utf-8') as f:
                            json.dump(cleaned, f, ensure_ascii=False, indent=2)
                except json.JSONDecodeError:
                    continue
    return sanitized_files



languages = ['en', 'es', 'pt', 'fr', 'ar']
for lang in languages:
    folder_path = os.path.join('../03_Outputs', 'SEA_Modules', lang)
    print(f"🔍 Scanning and sanitizing JSON files in {folder_path}...")
    results = validate_and_sanitize_json_files(folder_path)
    if results:
        print(f"✅ Sanitized files in [{lang}]:")
        for path, details in results:
            print(f" - {path}: replaced values {details}")
    else:
        print(f"✅ No invalid numeric values found in [{lang}]; all files are clean.")


🔍 Scanning and sanitizing JSON files in ../03_Outputs/SEA_Modules/en...
✅ Sanitized files in [en]:
 - ../03_Outputs/SEA_Modules/en/Module_4/4.1.4.json: replaced values $.segments[19].content.resources[0].href=nan; $.segments[19].content.resources[0].text=nan; $.segments[19].content.resources[1].href=nan; $.segments[19].content.resources[1].text=nan; $.segments[19].content.resources[2].href=nan; $.segments[19].content.resources[2].text=nan
🔍 Scanning and sanitizing JSON files in ../03_Outputs/SEA_Modules/es...
✅ No invalid numeric values found in [es]; all files are clean.
🔍 Scanning and sanitizing JSON files in ../03_Outputs/SEA_Modules/pt...
✅ No invalid numeric values found in [pt]; all files are clean.
🔍 Scanning and sanitizing JSON files in ../03_Outputs/SEA_Modules/fr...
✅ No invalid numeric values found in [fr]; all files are clean.
🔍 Scanning and sanitizing JSON files in ../03_Outputs/SEA_Modules/ar...
✅ No invalid numeric values found in [ar]; all files are clean.


In [9]:
# photo_ids = []
# for dirpath, _, filenames in os.walk("Well Designed+Source Diagrams"):
#     for filename in filenames:
#         if filename.endswith(".png"):
#             photo_id = filename[:-4]  # Remove ".png"
#             photo_ids.append(photo_id)
# print(photo_ids)